# Bank Customer Churn Prediction Using Artificial Neural Network (ANN)

**Aim:** Build a neural-network classifier to predict whether a bank customer will leave (`Exited = 1`) or stay (`Exited = 0`).

**Dataset:** `Churn_Modelling.csv` — 10,000 customer records.

Place the CSV file in the same folder as this notebook before running the cells.

## 1. Install dependencies (if required)
Run this cell only if the libraries are not already installed.

In [ ]:
# Uncomment and run if required
# !pip install tensorflow pandas numpy matplotlib scikit-learn

## 2. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## 3. Load and inspect the dataset

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")

print("Dataset shape:", df.shape)
display(df.head())

## 4. Check missing values and target distribution

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["Exited"].value_counts())

df["Exited"].value_counts().sort_index().plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Exited (0 = Stayed, 1 = Left)")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

## 5. Remove unnecessary columns
`RowNumber`, `CustomerId`, and `Surname` are identifiers, so they are excluded from model inputs.

In [ ]:
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

X = df.drop("Exited", axis=1)
y = df["Exited"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

## 6. Encode categorical features
One-hot encoding converts `Geography` and `Gender` into numeric columns.

In [ ]:
X = pd.get_dummies(
    X,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

# Keep the feature names for encoding future customer records
feature_columns = X.columns

print("Encoded features:")
print(X.head())
print("\nNumber of input features:", X.shape[1])

## 7. Split data into training and testing sets
80% of the records are used for training and 20% for testing. Stratification preserves the target-class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 8. Standardize the features
Fit the scaler only on training data, then apply the same transformation to test data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 9. Build the Artificial Neural Network
The network has two hidden layers (16 and 8 neurons) using ReLU, and one sigmoid output neuron for binary classification.

In [ ]:
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 10. Train the model
The validation split uses 20% of the training data for validation during training.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

## 11. Plot training and validation accuracy

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 12. Plot training and validation loss

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 13. Predict on test data

In [ ]:
y_probability = model.predict(X_test_scaled)
y_pred = (y_probability >= 0.5).astype(int).ravel()

print("First 10 predicted classes:", y_pred[:10])
print("First 10 actual classes:", y_test.iloc[:10].to_numpy())

## 14. Evaluate model accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", round(accuracy, 4))
print("Test Accuracy (%):", round(accuracy * 100, 2), "%")

## 15. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Stayed (0)", "Left (1)"])
plt.yticks([0, 1], ["Stayed (0)", "Left (1)"])
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

## 16. Classification report

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed (0)", "Left (1)"],
    zero_division=0
))

## 17. Predict churn for a new customer
The example customer below is encoded using the same feature columns and scaled using the already-fitted scaler.

In [ ]:
new_customer = pd.DataFrame({
    "CreditScore": [650],
    "Geography": ["Germany"],
    "Gender": ["Female"],
    "Age": [45],
    "Tenure": [5],
    "Balance": [120000],
    "NumOfProducts": [2],
    "HasCrCard": [1],
    "IsActiveMember": [0],
    "EstimatedSalary": [100000]
})

new_customer_encoded = pd.get_dummies(
    new_customer,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

# Ensure exactly the same input columns and order as during training
new_customer_encoded = new_customer_encoded.reindex(
    columns=feature_columns,
    fill_value=0
)

new_customer_scaled = scaler.transform(new_customer_encoded)

churn_probability = model.predict(new_customer_scaled)[0][0]

print("Churn probability:", round(float(churn_probability), 4))

if churn_probability >= 0.5:
    print("Prediction: Customer is likely to LEAVE.")
else:
    print("Prediction: Customer is likely to STAY.")

## Conclusion

The Artificial Neural Network learns patterns from customer demographic, financial, and banking-activity data to classify customers as likely to stay (`0`) or leave (`1`). Its performance is evaluated using accuracy, the confusion matrix, precision, recall, and F1-score.

**Note:** The model predicts a churn likelihood from the available dataset; the six-month timeframe is the problem's stated interpretation, not a separately measured time-to-churn label.